In [ ]:
# If needed, uncomment:
# !pip install -q pandas numpy matplotlib seaborn scipy

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")

# ================================================================
# 1. LOAD DATA
# ================================================================
file_path = "Table1.csv"
df = pd.read_csv(file_path, low_memory=False)

education_col = "Escolaridade"

infectious_cols = [
    c for c in df.columns
    if "doenças infecciosas" in c.lower()
    and "choice=" in c.lower()
    and "não" not in c.lower()
    and "outras" not in c.lower()
]

chronic_cols = [
    c for c in df.columns
    if "doenças crônicas" in c.lower()
    and "choice=" in c.lower()
    and "não" not in c.lower()
    and "outras" not in c.lower()
]

specialty_cols = [
    c for c in df.columns
    if ("encaminhamento" in c.lower() or "especialidade" in c.lower())
    and "choice=" in c.lower()
]

status_col = next(
    (c for c in df.columns if any(x in c.lower() for x in ["desfecho", "status", "atendimento"])),
    None
)

# ================================================================
# 2. TRANSLATION MAP — EDUCATION LABELS
# ================================================================
education_translation = {
    "Sem escolaridade": "No formal education",
    "Fundamental incompleto": "Incomplete elementary school",
    "Fundamental completo": "Complete elementary school",
    "Ensino médio incompleto": "Incomplete high school",
    "Ensino médio completo": "Complete high school",
    "Ensino superior incompleto": "Incomplete higher education",
    "Ensino superior completo ou mais": "Complete higher education or more",
    "Ignorado": "Unknown",
}

def translate_education(value):
    value_str = str(value).strip()
    return education_translation.get(value_str, value_str)

# ================================================================
# 3. LONGITUDINAL PROCESSING AND FILTERS
# ================================================================
df = df.sort_values(["Record ID", "Repeat Instance"], na_position="first")
cols_to_fill = [education_col] + infectious_cols + chronic_cols
df[cols_to_fill] = df.groupby("Record ID")[cols_to_fill].ffill()

df_visits = df[df["Repeat Instrument"].notna()].copy()
df_visits = df_visits.dropna(subset=[education_col])
df_visits = df_visits[~df_visits[education_col].astype(str).str.contains("Ignorado", case=False, na=False)]

df_visits["Education_Level_EN"] = df_visits[education_col].apply(translate_education)

# ================================================================
# 4. METRIC CALCULATION
# ================================================================
df_visits["Infectious_Disease_Burden"] = (df_visits[infectious_cols] == "Checked").sum(axis=1)
df_visits["Chronic_Disease_Burden"] = (df_visits[chronic_cols] == "Checked").sum(axis=1)

if status_col:
    df_visits["Absenteeism_Rate_Percent"] = (
        df_visits[status_col]
        .astype(str)
        .str.contains("não compareceu|falta", case=False, na=False)
        .astype(int) * 100
    )
else:
    df_visits["Absenteeism_Rate_Percent"] = 0

df_visits["Specialty_Referral_Percent"] = (
    (df_visits[specialty_cols] == "Checked").any(axis=1).astype(int) * 100
)

# ================================================================
# 5. CONSOLIDATED BASE TABLE
# ================================================================
metrics = [
    "Infectious_Disease_Burden",
    "Chronic_Disease_Burden",
    "Absenteeism_Rate_Percent",
    "Specialty_Referral_Percent",
]

summary_stats = []

for metric in metrics:
    stats_metric = df_visits.groupby("Education_Level_EN")[metric].agg(["mean", "std", "count"]).reset_index()
    stats_metric["sem"] = stats_metric["std"] / np.sqrt(stats_metric["count"])
    stats_metric["ci_95_margin"] = stats_metric["sem"] * 1.96
    stats_metric["Metric"] = metric
    summary_stats.append(stats_metric)

df_visual_base = pd.concat(summary_stats, ignore_index=True)

processed_file = "Processed_Article_Base_EN.csv"
descriptive_file = "Descriptive_Table_Education_EN.csv"

df_visits.to_csv(processed_file, index=False, encoding="utf-8-sig")
df_visual_base.to_csv(descriptive_file, index=False, encoding="utf-8-sig")

# ================================================================
# 6. VISUALIZATION
# ================================================================
education_order = (
    df_visits.groupby("Education_Level_EN")["Infectious_Disease_Burden"]
    .mean()
    .sort_values(ascending=False)
    .index
)

metric_titles = {
    "Infectious_Disease_Burden": "Infectious Disease Burden",
    "Chronic_Disease_Burden": "Chronic Disease Burden",
    "Absenteeism_Rate_Percent": "Absenteeism Rate (%)",
    "Specialty_Referral_Percent": "Specialty Referrals (%)",
}

metric_subtitles = {
    "Infectious_Disease_Burden": "Comparison across education levels",
    "Chronic_Disease_Burden": "Comparison across education levels",
    "Absenteeism_Rate_Percent": "Comparison across education levels",
    "Specialty_Referral_Percent": "Comparison across education levels",
}

plt.figure(figsize=(16, 12))
sns.set_theme(style="whitegrid")

colors = ["#c0392b", "#2980b9", "#8e44ad", "#27ae60"]

for i, metric in enumerate(metrics):
    plt.subplot(2, 2, i + 1)

    sns.pointplot(
        data=df_visits,
        x="Education_Level_EN",
        y=metric,
        order=education_order,
        color=colors[i],
        join=False,
        capsize=0.15,
        errorbar=("ci", 95),
    )

    groups = [g[metric].values for _, g in df_visits.groupby("Education_Level_EN") if len(g) > 1]
    f_stat, p_val = stats.f_oneway(*groups) if len(groups) > 1 else (0, 1)

    plt.title(
        f"{metric_titles[metric]}\n"
        f"{metric_subtitles[metric]} | ANOVA p-value: {p_val:.4f}",
        fontsize=13,
        fontweight="bold",
    )
    plt.xlabel("Education Level")
    plt.ylabel(metric_titles[metric])
    plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()

# ================================================================
# 7. CONSOLE REPORT
# ================================================================
print(f"\n{'=' * 80}")
print("DATASET GENERATED SUCCESSFULLY")
print("=" * 80)
print(f"1. '{processed_file}' (processed individual-level data)")
print(f"2. '{descriptive_file}' (means and 95% confidence intervals used in the chart)")